# L32 · 奖励模型：教 AI 分辨好坏

**学习目标**
- 理解「奖励模型（Reward Model, RM）」：给回答打分的评委
- 理解「偏好数据」：人类更爱哪个回答（A 优于 B）
- 用 numpy 训练一个 RM，学会「好回答高分、坏回答低分」

**前置依赖**：L31（SFT）、L11（梯度下降）  
**预计时长**：55 分钟  
**技术栈**：`numpy`、`matplotlib`（离线可运行）

---

## 概念讲解：奖励模型 = AI 选美评委

我们希望 AI 回答「又对又好」。但「好」很难写规则，所以**让人类标注偏好**：
「对同一个问题，回答 A 比回答 B 更好」。

**奖励模型** 就是学这个判分的评委：输入一个回答，输出一个分数。
训练目标（Bradley-Terry）：让「人类偏好的回答」得分显著高于「不被偏好的」。
它是下一步 PPO/RLHF 的「指挥棒」。

## 第一步：构造偏好数据（哪个回答更好）

In [ ]:
import numpy as np
np.random.seed(1)
# 每个回答用 3 维特征表示（如：相关度、礼貌、准确）
# 好回答特征高，坏回答特征低；人类偏好「好 > 坏」
good = np.random.randn(60, 3) + 1.0
bad = np.random.randn(60, 3) - 1.0
pairs = [(g, b) for g, b in zip(good, bad)]     # (偏好回答, 被否回答)
print("偏好样本：", len(pairs), "对，每对标注『好回答 vs 坏回答』")

## 第二步：训练奖励模型（Bradley-Terry 损失）

In [ ]:
W = np.random.randn(3) * 0.1
lr = 0.1
losses = []
for step in range(300):
    total_loss = 0
    grad = np.zeros(3)
    for g, b in pairs:
        r_g = W @ g          # 好回答得分
        r_b = W @ b          # 坏回答得分
        # 希望 r_g 远大于 r_b → sigmoid(r_g - r_b) 接近 1
        p = 1 / (1 + np.exp(-(r_g - r_b)))
        loss = -np.log(p + 1e-8)
        total_loss += loss
        grad += (p - 1) * (g - b)
    losses.append(total_loss / len(pairs))
    W -= lr * grad / len(pairs)
print(f"奖励模型训练完：损失 {losses[0]:.3f} → {losses[-1]:.3f}")
print(f"最终权重（学会了『好』长什么样）：{np.round(W, 2)}")

# 🎯 AHA 顿悟单元格：看评委「学会分辨好坏」

运行下面代码。你会看到：训练前后，奖励模型对「好回答 vs 坏回答」的**得分差距**从混乱变得清晰——
损失曲线下降，且好回答平均分显著高于坏回答。画一张对比柱状图。

> 你刚训练的，就是 RLHF 里的「奖励模型」。ChatGPT 的「好」，一半是这位评委教出来的。

In [ ]:
# ===== 运行我！看奖励模型学会打分 =====
import matplotlib.pyplot as plt
good_scores = [W @ g for g, _ in pairs]
bad_scores = [W @ b for _, b in pairs]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, color="#d62728"); ax[0].set_title("奖励模型训练损失")
ax[0].set_xlabel("步"); ax[0].set_ylabel("损失")
ax[1].bar(["好回答", "坏回答"], [np.mean(good_scores), np.mean(bad_scores)],
          color=["#2ca02c", "#ff7f0e"])
ax[1].set_title(f"打分对比（差距 {np.mean(good_scores)-np.mean(bad_scores):.2f}）")
ax[1].set_ylabel("平均奖励分")
plt.tight_layout(); plt.show()
print(f"  🏆 好回答均分 {np.mean(good_scores):.2f}  >  坏回答 {np.mean(bad_scores):.2f}")
print("  ✨ 你的『AI 评委』已学会分辨好坏，下一步它就能指挥模型变好！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：Bradley-Terry 配对偏好损失；sigmoid + 交叉熵；「学到的是相对偏好而非绝对分」。  
**易错点**：数值溢出（sigmoid 已用指数，需保证稳定；loss 加 1e-8 防 log(0)）；梯度符号（p-1 使偏好样本推高好回答）。  
**AHA 机制**：损失曲线 + 好坏得分对比柱状图，强「评委长大」实感。  
**衔接**：L33 DPO（绕过显式 RM）；L34 PPO（用 RM 当奖励信号）；L35 RLHF 串联。  
**依赖**：`pip install numpy matplotlib`。  
**真 LLM 路径**：真实 RM 是在 SFT 模型上加一个标量头，训练用 paired preference 数据（如 Anthropic HH、UltraFeedback）。

# 📚 作业 / 下一步

1. 把 `lr` 改成 1.0，看损失是否震荡。
2. 思考：如果人类标注本身有偏见，奖励模型会学到什么？（引出 RLHF 偏见问题）
3. 下一课 **L33 DPO：最简单的高效对齐** —— 不用显式奖励模型，一步到位对齐。